# Lab 16-09 · Changing Sample Rates Without Damage

Five parts, each built on `multirate.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math, os, subprocess, sys, tempfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import multirate as mr
from make_vectors import tests, build, N, R, W
sys.path.insert(0, "../16-06-lab")
import firdesign as fd

def tone_peaks(y, fs, k=3):
    Y = np.abs(np.fft.rfft(y * np.hanning(len(y))))
    f = np.fft.rfftfreq(len(y), 1 / fs)
    idx = []
    for _ in range(k):
        i = int(np.argmax(Y)); idx.append(f[i]); Y[max(0, i - 8): i + 9] = 0
    return sorted(round(float(v) / 1000, 1) for v in idx)

## Part 1 · Aliasing and imaging

*Predict:* where do 10 kHz and 17 kHz land after decimating 48 kHz by 4 with no filter?

In [2]:
n = np.arange(48000)
x = np.cos(2 * np.pi * 1000 * n / 48000) + np.cos(2 * np.pi * 10000 * n / 48000) + np.cos(2 * np.pi * 17000 * n / 48000)
pk = tone_peaks(mr.downsample(x, 4), 12000)
print("after decimating by 4 (kHz):", pk)
assert pk == [1.0, 2.0, 5.0]
x16 = np.cos(2 * np.pi * 5000 * np.arange(16000) / 16000)
pk = tone_peaks(mr.upsample(x16, 3), 48000)
print("after upsampling by 3 (kHz):", pk)
assert pk == [5.0, 11.0, 21.0]

after decimating by 4 (kHz): [1.0, 2.0, 5.0]
after upsampling by 3 (kHz): [5.0, 11.0, 21.0]


## Part 2 · Polyphase equivalence

*Predict:* by how much do the polyphase outputs differ from the direct ones?

In [3]:
rng = np.random.default_rng(16)
x = rng.standard_normal(3000)
h = rng.standard_normal(96)
for M in (2, 3, 8):
    d = np.max(np.abs(mr.decimate_direct(x, h, M) - mr.decimate_polyphase(x, h, M)))
    i = np.max(np.abs(mr.interpolate_direct(x[:500], h, M) - mr.interpolate_polyphase(x[:500], h, M)))
    print(f"M = L = {M}: decimator difference {d:.1e}, interpolator difference {i:.1e}")
    assert d < 1e-10 and i < 1e-10
print("decimating 48 kHz by 8 with 96 taps:", mr.mults_per_second(96, 48000), "direct,", mr.mults_per_second(96, 6000), "polyphase")

M = L = 2: decimator difference 7.1e-15, interpolator difference 3.6e-15
M = L = 3: decimator difference 7.1e-15, interpolator difference 3.6e-15
M = L = 8: decimator difference 7.1e-15, interpolator difference 3.6e-15
decimating 48 kHz by 8 with 96 taps: 4608000 direct, 576000 polyphase


## Part 3 · A multistage decimator

*Predict:* the worst alias that lands on 0–20 kHz, in dB.

In [4]:
stages = []
for F, fp in ((384e3, 20e3), (192e3, 20e3), (96e3, 20e3)):
    fst = F / 2 - fp
    for K in range(40):
        n_ = 4 * K + 3
        h = signal.remez(n_, [0, fp, fst, F / 2], [1, 0], fs=F, grid_density=64)
        m = fd.measure(h, 2 * np.pi * fp / F, 2 * np.pi * fst / F)
        if m[0] <= 1e-4 and m[1] <= 1e-4:
            break
    stages.append(h)
print("stage lengths:", [len(s) for s in stages])
assert [len(s) for s in stages] == [11, 15, 59]
f = np.linspace(0, 192e3, 200001)
w = 2 * np.pi * f / 384e3
H = np.ones_like(w, dtype=complex)
for s, scale in zip(stages, (1, 2, 4)):
    H = H * signal.freqz(s, worN=scale * w)[1]
H = np.abs(H)
pb = f <= 20e3
alias = np.zeros_like(f, bool)
for k in range(1, 5):
    alias |= np.abs(f - k * 48e3) <= 20e3
rip = (20 * math.log10(H[pb].max()), 20 * math.log10(H[pb].min()))
att = -20 * math.log10(H[alias].max())
print(f"passband ripple {rip[1]:+.4f} to {rip[0]:+.4f} dB; worst alias {att:.1f} dB down")
assert max(abs(rip[0]), abs(rip[1])) < 0.0011 and abs(att - 84.3) < 0.05
plt.figure(figsize=(6, 3)); plt.plot(f / 1e3, 20 * np.log10(np.maximum(H, 1e-9))); plt.ylim(-140, 5)
plt.xlabel("kHz at 384 kHz"); plt.ylabel("dB"); plt.show()

stage lengths: [11, 15, 59]
passband ripple -0.0010 to +0.0010 dB; worst alias 84.3 dB down


## Part 4 · 44.1 kHz to 48 kHz

*Predict:* how many taps of the 3201-tap prototype does each output sample use?

In [5]:
L, M = 160, 147
Np = 3201
h = signal.firwin(Np, 20000, window=("kaiser", 9.0), fs=44100 * L) * L
x = np.sin(2 * np.pi * 1000 * np.arange(4410) / 44100)
y = mr.resample_polyphase(x, h, L, M)
delay = (Np - 1) / 2 / M
m = np.arange(len(y))
ref = np.sin(2 * np.pi * 1000 * (m - delay) / 48000)
err = np.max(np.abs(y[500:-100] - ref[500:-100]))
print(f"{-(-Np // L)} taps per output; peak error {20 * math.log10(err):.1f} dB")
assert abs(20 * math.log10(err) + 95.8) < 0.05

21 taps per output; peak error -95.8 dB


## Part 5 · The CIC decimator

*Predict:* the output for a full-scale DC input if the integrators saturate instead of wrapping.

In [6]:
lines = build()
assert open("vectors.hex").read().split() == lines, "vectors.hex is stale: rerun make_vectors.py"
y = mr.cic_decimate_int(tests[0], R, N, W)
print("wrap-around, +2047 DC: last outputs", y[-3:], " expected", 2047 * R ** N)
assert y[-1] == 2047 * R ** N

def cic_saturating(x, R, N, width):
    lo, hi = -(1 << (width - 1)), (1 << (width - 1)) - 1
    integ, combs, out = [0] * N, [0] * N, []
    for n_, xn in enumerate(x):
        acc = xn
        for i in range(N):
            integ[i] = min(hi, max(lo, integ[i] + acc)); acc = integ[i]
        if n_ % R == R - 1:
            v = acc
            for i in range(N):
                v, combs[i] = min(hi, max(lo, v - combs[i])), v
            out.append(v)
    return out
ys = cic_saturating(tests[0], R, N, W)
print("saturating integrators: last outputs", ys[-3:])
assert ys[-1] != 2047 * R ** N
try:
    vvp = os.path.join(tempfile.mkdtemp(), "cic.vvp")
    subprocess.run(["iverilog", "-g2012", "-o", vvp, "cic_decim_tb.v", "cic_decim.v"], check=True, capture_output=True)
    out = subprocess.run(["vvp", vvp], capture_output=True, text=True).stdout
    print([ln for ln in out.splitlines() if "TEST" in ln][0]); assert "TEST PASSED" in out
except FileNotFoundError:
    print("Icarus Verilog not installed here; CI runs the testbench (tools/run_hdl.py).")

wrap-around, +2047 DC: last outputs [8384512, 8384512, 8384512]  expected 8384512
saturating integrators: last outputs [-454435, 0, 0]
TEST PASSED (12 sequences, 8 outputs each, bit-exact)
